# AgentArk Model Replay Tutorial

AgentArk evaluation records are deterministic enough to replay saved action trajectories against the same task and seed. This Colab clones AgentArk from GitHub, downloads the Linux runtime, fetches a public Snake replay JSONL from Hugging Face, and visualizes one saved trajectory.


In [ ]:
# @title Step 1: Install AgentArk and download the Linux runtime { display-mode: "form" }
# @markdown Clones the public AgentArk repository, creates a Python 3.10 virtual environment, downloads the Unity runtime, and installs AgentArk in editable mode.

AGENTARK_REPO_URL = "https://github.com/P90-RushB/AgentArk.git"  # @param {type:"string"}
AGENTARK_BRANCH = "main"  # @param {type:"string"}
ENV_ZIP_URL = "https://huggingface.co/datasets/P90-RushB/AgentArk/resolve/main/artifacts/envs/1.0.1/linux/AgentArk-env-1.0.1-linux.zip?download=true"  # @param {type:"string"}
FORCE_REINSTALL = False  # @param {type:"boolean"}

import shutil
import subprocess
import sys
from pathlib import Path

%cd /content

CONTENT_ROOT = Path("/content")
AGENTARK_ROOT = CONTENT_ROOT / "AgentArk"
VENV_DIR = CONTENT_ROOT / "agentark_env"
VENV_PY = VENV_DIR / "bin" / "python"
ENV_ZIP = CONTENT_ROOT / "AgentArk-env-1.0.1-linux.zip"
ENV_ROOT = CONTENT_ROOT / "AgentArk-env-1.0.1-linux"


def run(cmd):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True)


print("Installing OS packages...")
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "git", "unzip", "wget", "xvfb", "python3.10", "python3.10-venv"])

if FORCE_REINSTALL:
    print("Removing previous /content installation...")
    shutil.rmtree(AGENTARK_ROOT, ignore_errors=True)
    shutil.rmtree(VENV_DIR, ignore_errors=True)
    shutil.rmtree(ENV_ROOT, ignore_errors=True)
    if ENV_ZIP.exists():
        ENV_ZIP.unlink()

if not VENV_DIR.exists():
    run(["python3.10", "-m", "venv", VENV_DIR])
else:
    print("Using existing virtual environment:", VENV_DIR)

if not AGENTARK_ROOT.exists():
    branch = AGENTARK_BRANCH.strip()
    clone_cmd = ["git", "clone", "--depth", "1"]
    if branch:
        clone_cmd.extend(["--branch", branch])
    clone_cmd.extend([AGENTARK_REPO_URL, AGENTARK_ROOT])
    run(clone_cmd)
else:
    print("Using existing AgentArk checkout:", AGENTARK_ROOT)

if not (AGENTARK_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(f"Expected {AGENTARK_ROOT / 'pyproject.toml'} after cloning AgentArk")

if not ENV_ROOT.exists():
    run(["wget", "-q", "-O", ENV_ZIP, ENV_ZIP_URL])
    run(["unzip", "-q", "-o", ENV_ZIP, "-d", CONTENT_ROOT])
else:
    print("Using existing Unity runtime:", ENV_ROOT)

exe = ENV_ROOT / "AgentArk.x86_64"
if not exe.exists():
    raise FileNotFoundError(f"Expected Unity executable at {exe}")
exe.chmod(0o755)

print("Installing AgentArk Python package...")
run([VENV_PY, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run([VENV_PY, "-m", "pip", "install", "-q", "-e", AGENTARK_ROOT])

print("Installing notebook helper packages...")
run([sys.executable, "-m", "pip", "install", "-q", "omegaconf"])

print("AgentArk root:", AGENTARK_ROOT)
print("Unity runtime:", ENV_ROOT)
print("Python:", VENV_PY)


## Configure Runtime And Preview Tasks

Step 2 writes `/content/AgentArk/.env`, enables the Colab virtual display mode in the Unity runtime config, and prints the tasks included in the downloaded runtime.


In [ ]:
# @title Step 2: Configure runtime paths and show available tasks { display-mode: "form" }

import os
from pathlib import Path

from omegaconf import OmegaConf

AGENTARK_ROOT = Path("/content/AgentArk")
ENV_ROOT = Path("/content/AgentArk-env-1.0.1-linux")
MOD_PATH = ENV_ROOT / "AgentArk_Data" / "Resources" / "Mods"
TASK_STORE_PATH = MOD_PATH / "all_tasks"
RUNTIME_POOL_ROOT = Path("/tmp/agentark_runtime_pool")
VENV_PY = "/content/agentark_env/bin/python"

if not (AGENTARK_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run Step 1 first; /content/AgentArk is missing.")
if not (ENV_ROOT / "AgentArk.x86_64").exists():
    raise RuntimeError("Run Step 1 first; the AgentArk Unity runtime is missing.")

env_values = {
    "AGENTARK_ENV_PATH": str(ENV_ROOT / "AgentArk.x86_64"),
    "AGENTARK_MOD_PATH": str(MOD_PATH),
    "AGENTARK_TASK_STORE_PATH": str(TASK_STORE_PATH),
    "AGENTARK_RUNTIME_TEMPLATE_ROOT": str(ENV_ROOT),
    "AGENTARK_RUNTIME_POOL_ROOT": str(RUNTIME_POOL_ROOT),
    "MLAGENTS_PYTHON_BIN": VENV_PY,
}
os.environ.update(env_values)

(AGENTARK_ROOT / ".env").write_text(
    "\n".join(f"{key}={value}" for key, value in env_values.items()) + "\n",
    encoding="utf-8",
)

mod_config = MOD_PATH / "config.yaml"
if mod_config.exists():
    mod_conf = OmegaConf.load(str(mod_config))
    mod_conf.virtual_display = True
    OmegaConf.save(config=mod_conf, f=str(mod_config))
else:
    print("Warning: mod config not found:", mod_config)

print("Wrote:", AGENTARK_ROOT / ".env")
print("Available tasks:")
task_options = sorted(path.name for path in TASK_STORE_PATH.iterdir() if path.is_dir()) if TASK_STORE_PATH.exists() else []
for name in task_options[:80]:
    print("-", name)
if len(task_options) > 80:
    print(f"... and {len(task_options) - 80} more")
if not task_options:
    print("No tasks found. Check AGENTARK_MOD_PATH and the downloaded runtime.")


In [ ]:
# @title Step 3: Download a Snake replay JSONL { display-mode: "form" }

REPLAY_URL = "https://huggingface.co/datasets/P90-RushB/AgentArk/resolve/main/artifacts/records/0000-0099/0001-Snake/Snake_seeds_1_10.jsonl?download=true"  # @param {type:"string"}
REPLAY_JSONL_PATH = "/content/Snake_seeds_1_10.jsonl"  # @param {type:"string"}

import subprocess
from pathlib import Path

path = Path(REPLAY_JSONL_PATH)
path.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(["wget", "-q", "-O", str(path), REPLAY_URL], check=True)
line_count = sum(1 for _ in path.open("r", encoding="utf-8"))
print("Downloaded:", path)
print("JSONL rows:", line_count)


## Replay The Snake Example

Step 4 replays one row from the downloaded JSONL. Change `REPLAY_RECORD_INDEX` to inspect a different saved model/seed trajectory. The Snake leaderboard and task notes are available on the [AgentArk Hub](https://p90-rushb.github.io/agentark-hub/tasks/snake/).


In [ ]:
# @title Step 4: Replay a saved model trajectory { display-mode: "form" }

TARGET_TASK_NAME = "Snake"  # @param ["AxleBoardAlignment3D", "BlockWorldPathCopy3D", "ColorFovCount", "CraneStackTower2D", "DelayTrain", "DoorSpin3Turns", "FishingJoy2D", "FlappyBird", "GoldMiner2D", "GrenadeClusterCalibration3D", "MarbleStop", "Match3ScoreGoal2D", "MirrorRelay2D", "ObjectRotationMatch", "Pushbox", "Reach2048Tile2D", "RotationSpeedSort3D", "Snake", "SpatialRelationAxisOrder3D", "SpatialRelationMatch3D", "StarterRouteJump3D", "StickBridgeEstimate2D", "Tetris", "TetrisHard", "TightropeSprint3D", "ZigzagPillarJump3D"] {allow-input: true}
REPLAY_JSONL_PATH = "/content/Snake_seeds_1_10.jsonl"  # @param {type:"string"}
REPLAY_RECORD_INDEX = 0  # @param {type:"integer"}
STEP_DELAY_S = 0.5  # @param {type:"number"}
REQUIRE_MATCH = True  # @param {type:"boolean"}
VIEWER_PORT = 18181  # @param {type:"integer"}

import os
import subprocess
import time
from pathlib import Path

from google.colab import output
from omegaconf import OmegaConf

AGENTARK_ROOT = Path("/content/AgentArk")
CONFIG_PATH = AGENTARK_ROOT / "config" / "ark_env" / "replay.example.yaml"
VENV_PY = "/content/agentark_env/bin/python"
LOG_PATH = Path("/content/agentark_replay.log")

if not os.getenv("AGENTARK_ENV_PATH"):
    raise RuntimeError("Run Step 2 first so AgentArk runtime paths are available.")
if not Path(REPLAY_JSONL_PATH).exists():
    raise FileNotFoundError(f"Replay JSONL not found: {REPLAY_JSONL_PATH}. Run Step 3 first or update the path.")


def ensure_section(parent, name):
    if name not in parent or parent[name] is None:
        parent[name] = {}
    return parent[name]


conf = OmegaConf.load(str(CONFIG_PATH))
env_cfg = ensure_section(conf, "env_cfg")
env_overrides = ensure_section(env_cfg, "env_config_overrides")
env_overrides.virtual_display = True
env_overrides.num_parallel_envs = 1

replay_cfg = ensure_section(conf, "replay")
replay_cfg.records_path = REPLAY_JSONL_PATH
replay_cfg.record_index = int(REPLAY_RECORD_INDEX)
replay_cfg.task_name = TARGET_TASK_NAME
replay_cfg.step_delay_s = float(STEP_DELAY_S)
replay_cfg.require_match = bool(REQUIRE_MATCH)

hooks = ensure_section(conf, "hooks")
visualization = ensure_section(hooks, "visualization")
visualization.enabled = True
visualization.host = "127.0.0.1"
visualization.port = int(VIEWER_PORT)
visualization.open_browser = True
visualization.keep_open_on_end = True
human = ensure_section(hooks, "human_interaction")
human.enabled = False

OmegaConf.save(config=conf, f=str(CONFIG_PATH))

if "agent_process" in globals() and agent_process.poll() is None:
    print("Stopping previous AgentArk process...")
    agent_process.terminate()
    agent_process.wait(timeout=20)

cmd = [VENV_PY, "-m", "agent_ark.ark_eval.run_replay", "--config", "config/ark_env/replay.example.yaml"]
log_file = LOG_PATH.open("w", encoding="utf-8")
agent_process = subprocess.Popen(
    cmd,
    cwd=str(AGENTARK_ROOT),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)

time.sleep(3)
if agent_process.poll() is not None:
    print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-5000:])
    raise RuntimeError("AgentArk replay exited early; see the log above.")

print("Replay running in background.")
print("Log file:", LOG_PATH)
print("Open visualization:")
print(output.eval_js(f"google.colab.kernel.proxyPort({int(VIEWER_PORT)})"))
